# Prize-Sizing: is there a cooling-energy prize to win?

**Question this notebook answers (and *only* this):** how much controllable cooling energy is recoverable on the ExaDigiT FMU, against a realistic baseline, on the real load trace — *before* investing further in workload synthesis / RL / GNN. Method-agnostic: we bound the prize with a baseline and a strong reference controller; we do NOT test a learned method here.

**Objective** (single source of truth in `rollout.POWER_VARS`):
`P_cooling = W_flow_CT + W_flow_CTWP + W_flow_HTWP + Σ_k W_flow_CDUP`  (watts; η=0.85 baked in → input power). `E = Σ P·Δt`.

**Decision rule** (fractional `ΔE = (E_base − E_ref)/E_base`):
- `ΔE ≈ 1–3%`  → baseline already near-optimal; thesis in trouble.
- `ΔE ≈ 5–15%` → real & publishable; synthesis/RL justified.
- `ΔE > 20%`  → suspect a mis-defined objective (e.g. pump term) before celebrating.

**We size the prize under BOTH exogenous pipelines and compare:**
- **v2** = project/paper standard (all sustain-lc train/eval use it) → ΔE comparable to their baselines, BUT v2 flattens load (pins per-cabinet total to a constant) → **conservatively understates** the prize; a small v2 ΔE is likely a flattening artifact.
- **v1** = magnitude-preserving (uses only 5 of 25 CDU cols) → where the **load-following** prize actually shows up.

**Assumptions to verify before trusting numbers** (see code comments):
1. `POWER_VARS` names match the FMU (Step 0 checks this).
2. `T_MAX_K` — the cabinet temperature limit — is a **placeholder**; set it to the real spec.
3. `policies.BASELINE_CDU` — the baseline CDU setpoints — is a **placeholder** (range midpoints); set to true current practice.

In [ ]:
# Run this notebook from the validation/ directory so `import rollout` resolves.
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import rollout, policies
importlib.reload(rollout); importlib.reload(policies)

# ---- experiment constants ----
STEP_SIZE = 15.0          # s  (FMU ZOH step)

# !! ASSUMPTION TO VERIFY !!  cabinet temperature limit (performance/safety constraint).
# Placeholder: 45 °C. Replace with the real CDU/cabinet spec before trusting feasibility.
T_MAX_K = 273.15 + 45.0

# Static-setpoint sweep grid (held constant over a run). Coarse by default: each grid point
# is a full rollout and v2 passes are 120 h. Refine once a version looks promising.
tsec_grid = np.linspace(-1, 1, 3)   # CDU supply-temp knob
dp_grid   = np.linspace(-1, 1, 3)   # pump dp knob
ct_grid   = [2, 4, 6]               # CT approach: colder / rule / warmer

# QUICK_HOURS: cap each rollout for a fast first look (partial pass). Set to None for a
# FULL pass (v1 = 24 h, v2 = 120 h) before trusting numbers. A partial v2 pass covers only
# the first folded block(s).
QUICK_HOURS = None
def _stop_time(v):
    return QUICK_HOURS * 3600 if QUICK_HOURS else rollout.full_pass_stop_time(v, STEP_SIZE)

print('T_MAX_K =', T_MAX_K, '| grid points =', len(tsec_grid)*len(dp_grid)*len(ct_grid),
      '| pass h: v1=%.0f v2=%.0f' % (_stop_time(1)/3600, _stop_time(2)/3600))

## Step 0 — verify the objective variable names
If `compute_P_cooling` raises a KeyError, the printed `W_flow` list shows the real names — correct `rollout.POWER_VARS` accordingly.

In [ ]:
_probe = rollout.make_env(exogen_gen_v=2)
_probe.reset()
print('FMU W_flow variables found:')
for n in rollout.list_power_vars(_probe):
    print('  ', n)
print('\nP_cooling at t=0:', rollout.compute_P_cooling(_probe.fmu), 'W')
del _probe

## Step 1 — inspect v1 vs v2 preprocessing
Plot one cabinet's load (3 branches + total) under each pipeline, straight from the generator (no FMU run). Expectation from the code: **v1 total swings** (idle/busy), **v2 total is ~flat** because the `softmax × global-max` rescale pins each cabinet's 3 branches to sum to a constant peak. The printed CV (std/mean of the total) quantifies it — v2 ≈ 0 confirms the flattening.

In [ ]:
tr1 = rollout.exogenous_trace(1)   # (5761, 16):  cols 0..14 = 5 cab x 3 branch, col 15 = wetbulb
tr2 = rollout.exogenous_trace(2)   # (~28805, 16)
h1 = np.arange(len(tr1)) * STEP_SIZE / 3600
h2 = np.arange(len(tr2)) * STEP_SIZE / 3600

fig, ax = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for a, tr, h, lab in [(ax[0], tr1, h1, 'v1'), (ax[1], tr2, h2, 'v2')]:
    a.plot(h, tr[:, 0:3].sum(axis=1) / 1e3, 'k', lw=1.2, label='cabinet-0 total')
    for b in range(3):
        a.plot(h, tr[:, b] / 1e3, lw=0.6, alpha=0.6)
    a.set_title(f'{lab}: cabinet-0 load'); a.set_xlabel('hour'); a.legend()
ax[0].set_ylabel('kW')
plt.tight_layout(); plt.show()

cv = lambda x: x.std() / x.mean()
print('cabinet-0 TOTAL load CV  ->  v1: %.3f   v2: %.3f   (v2 ~0 confirms flattening)'
      % (cv(tr1[:, 0:3].sum(1)), cv(tr2[:, 0:3].sum(1))))

## Step 2 — size the prize for both v1 and v2
Under each pipeline: run the baseline (CT on the wetbulb+10°F rule, CDU nominal) and a static-setpoint sweep; the gap baseline→best-feasible-static is the ΔE **floor**. `size_prize(v)` returns the energies, the floor, and the raw frames. We run it for both v1 and v2 and compare.

In [ ]:
def verdict(dE):
    if not np.isfinite(dE): return 'NO FEASIBLE STATIC POINT (loosen grid / check T_MAX_K)'
    if dE < 0.03:  return 'TROUBLE: baseline near-optimal'
    if dE < 0.05:  return 'MARGINAL: check robustness'
    if dE <= 0.20: return 'PRIZE EXISTS'
    return 'SUSPECT >20%: check objective / T_max / baseline'

def size_prize(exogen_gen_v):
    run_kw = dict(stop_time=_stop_time(exogen_gen_v), step_size=STEP_SIZE,
                  exogen_gen_v=exogen_gen_v)
    df_base = rollout.run_policy(policies.baseline_policy, name='baseline', **run_kw)
    sb = rollout.summarize(df_base, T_MAX_K, STEP_SIZE)
    recs = []
    for ta in tsec_grid:
        for da in dp_grid:
            for ct in ct_grid:
                pol = policies.make_constant_policy(policies.make_cdu_vec(ta, da), ct_action=ct)
                df  = rollout.run_policy(pol, name='static', **run_kw)
                s   = rollout.summarize(df, T_MAX_K, STEP_SIZE)
                s.update(tsec_a=ta, dp_a=da, ct=ct)
                recs.append(s)
    sweep = pd.DataFrame(recs)
    feas  = sweep[sweep['feasible']]
    E_base   = sb['E_cooling_J']
    E_static = feas['E_cooling_J'].min() if len(feas) else np.nan
    dE_floor = (E_base - E_static) / E_base if len(feas) else np.nan
    return dict(exogen_v=exogen_gen_v, stop_h=run_kw['stop_time'] / 3600,
                E_base_J=E_base, E_static_J=E_static, dE_floor=dE_floor,
                base_feasible=sb['feasible'], base_margin_K=sb['T_cab_margin_K'],
                n_feasible=len(feas), n_total=len(sweep), df_base=df_base, sweep=sweep)

In [ ]:
# Runs baseline + sweep for BOTH pipelines. SLOW under v2 (120 h/pass). Set QUICK_HOURS in
# the constants cell for a fast first look, or run a single version: res = {2: size_prize(2)}.
res = {v: size_prize(v) for v in (1, 2)}
print('done:', {v: f"{res[v]['n_feasible']}/{res[v]['n_total']} feasible" for v in res})

In [ ]:
# Baseline cooling power + peak cabinet temp, v1 vs v2 overlay.
fig, ax = plt.subplots(2, 1, figsize=(11, 6))
for v, c in [(1, 'tab:blue'), (2, 'tab:orange')]:
    d = res[v]['df_base']
    ax[0].plot(d['t_s'] / 3600, d['P_cooling_W'] / 1e3, c, label=f'v{v}')
    ax[1].plot(d['t_s'] / 3600, d['T_cab_max_K'] - 273.15, c, label=f'v{v}')
ax[0].set_ylabel('P_cooling [kW]'); ax[0].set_title('Baseline cooling power'); ax[0].legend()
ax[1].axhline(T_MAX_K - 273.15, color='r', ls='--', label='T_max')
ax[1].set_ylabel('max cab T [°C]'); ax[1].set_xlabel('hour'); ax[1].set_title('Baseline peak cabinet temp'); ax[1].legend()
plt.tight_layout(); plt.show()

# Side-by-side ΔE floor.
summary = pd.DataFrame([{
    'exogen_v': r['exogen_v'], 'pass_h': r['stop_h'],
    'E_base_kWh': r['E_base_J'] / 3.6e6, 'E_static_kWh': r['E_static_J'] / 3.6e6,
    'dE_floor_%': r['dE_floor'] * 100, 'base_margin_K': r['base_margin_K'],
    'feasible': f"{r['n_feasible']}/{r['n_total']}", 'verdict': verdict(r['dE_floor']),
} for r in res.values()])
summary

## Step 3 — oracle ceiling (optional, slow)
Time-varying clairvoyant schedule = true upper bound, for ONE chosen pipeline. Set `ORACLE_V = 1` to bound the load-following prize (magnitude-preserving), or `2` for paper-comparability. Each candidate is a full rollout — keep `n_segments`/`maxiter` small and parallelize the inner rollouts before scaling up. Build this only after a floor looks promising.

In [ ]:
ORACLE_V   = 1      # which pipeline to bound (1 = load-following prize; 2 = paper-comparable)
RUN_ORACLE = False  # set True when ready (slow; a v2 candidate is a 120 h pass)
oracle = None
if RUN_ORACLE:
    best_params, df_oracle, r = policies.optimize_oracle(
        rollout.run_policy, rollout.summarize,
        n_segments=6, T_max_K=T_MAX_K, step_size=STEP_SIZE,
        stop_time=_stop_time(ORACLE_V), exogen_gen_v=ORACLE_V, maxiter=10)
    so = rollout.summarize(df_oracle, T_MAX_K, STEP_SIZE)
    E_base_v = res[ORACLE_V]['E_base_J']
    oracle = dict(v=ORACLE_V, dE_ceiling=(E_base_v - so['E_cooling_J']) / E_base_v, **so)
    print(oracle)

## Step 4 — verdict

In [ ]:
print('=== Prize-sizing: ΔE floor (best static vs baseline) ===')
for r in res.values():
    print('v%d  ΔE_floor = %6.2f%%   %-22s | E_base=%.1f kWh, pass=%.0fh'
          % (r['exogen_v'], r['dE_floor'] * 100, verdict(r['dE_floor']),
             r['E_base_J'] / 3.6e6, r['stop_h']))
if oracle is not None:
    print('\noracle (v%d)  ΔE_ceiling = %.2f%%' % (oracle['v'], oracle['dE_ceiling'] * 100))

print('\nReading guide:')
print(' - v1 preserves load magnitude -> the LOAD-FOLLOWING prize lives here.')
print(' - v2 is the project/paper standard but FLATTENS load (constant per-cabinet total),')
print('   so a small v2 ΔE is likely an artifact, not absence of a prize.')
print(' - ΔE is the FMU-measured fractional prize; eta cancels so it is unbiased. Real-world')
print('   transfer has two opposing out-of-model effects (do not adjust the threshold here).')